# ARC-AGI — Generate Rule Descriptions with Claude

## What this notebook does

Calls the Anthropic API to generate a structured natural-language description
of the transformation rule for each of the 400 ARC training tasks.
The descriptions are saved to `data/claude_descriptions.json` and later
merged into the fine-tuning dataset for our small GPT-2 model.

## Why we need this

Our current training data comes from the LARC dataset — human descriptions
of ARC tasks collected via crowdsourcing.  Many of those descriptions are
vague or incomplete (e.g. *"move the bars into colors"*), which means the
model cannot learn a reliable rule from them.

Claude produces descriptions like:

> **RULE:** The 3×3 input grid is used as a meta-pattern to tile a 9×9 output
> grid: each non-zero cell causes the entire input pattern to be placed in the
> corresponding 3×3 block of the output, while each zero cell causes that block
> to be filled with zeros.

That level of precision lets the small LLM (and eventually a grid-generator)
actually reconstruct the correct output.

## The two-stage pipeline

```
Training pairs
     │
     ▼
┌──────────────┐    rule description    ┌──────────────┐    predicted
│  Stage 1     │ ──────────────────────▶│  Stage 2     │ ──────────▶ output grid
│  Describer   │                        │  Generator   │
│  (GPT-2 FT)  │                        │  (GPT-2 FT)  │
└──────────────┘                        └──────────────┘
```

- **Stage 1** is trained on: (training pairs) → (rule description)
- **Stage 2** is trained on: (rule description + test input) → (output grid)
- At inference time, Stage 1 generates a hypothesis; Stage 2 generates a
  candidate grid; a verification step checks it against the training pairs.

This notebook generates the training data for Stage 1.

## Description format

Each description has four sections:

| Section | Purpose |
|---|---|
| `TYPE` | Task category — geometric / fill / count / extend / crop / other |
| `RULE` | One-sentence summary of the transformation |
| `STEPS` | Numbered procedural steps precise enough to reconstruct the output |
| `RELATIONSHIP` | What stays the same, what changes, any exceptions |

## Cost estimate

400 tasks × ~600 tokens in + ~400 tokens out × `claude-sonnet-4-6` ≈ **~$0.80**
total.  Run time on Colab (rate-limited): approximately 10–15 minutes.

In [ ]:
# ── Cell 1: Mount Google Drive ──────────────────────────────────────────
# Progress is saved here after every task so a disconnection loses nothing.
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/arc-agi'
os.makedirs(DRIVE_DIR, exist_ok=True)
DESCRIPTIONS_FILE = f'{DRIVE_DIR}/claude_descriptions.json'
print(f'Progress file: {DESCRIPTIONS_FILE}')

In [ ]:
# ── Cell 2: Clone repo + install Anthropic SDK + set API key ────────────
import os, subprocess, sys

def run(cmd): subprocess.run(cmd, check=True)

if not os.path.exists('/content/arc-agi-solver'):
    run(['git', 'clone', 'https://github.com/rodehyde/arc-agi-solver',
         '/content/arc-agi-solver'])
else:
    run(['git', '-C', '/content/arc-agi-solver', 'pull'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'anthropic'])

# Anthropic API key — store in Colab Secrets as 'ANTHROPIC_API_KEY'
# Tools → Secrets → Add new secret → Name: ANTHROPIC_API_KEY
from google.colab import userdata
try:
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('Anthropic API key set.')
except Exception as e:
    print(f'WARNING: No ANTHROPIC_API_KEY in Colab Secrets ({e})')
    print('Add it via Tools → Secrets before running Cell 4.')

In [ ]:
# ── Cell 3: Download ARC training data ──────────────────────────────────
import os, urllib.request, zipfile

ARC_DIR = '/content/arc-agi-solver/data/training'
os.makedirs(ARC_DIR, exist_ok=True)

if len(os.listdir(ARC_DIR)) < 400:
    print('Downloading ARC training data...')
    url = 'https://github.com/fchollet/ARC-AGI/archive/refs/heads/master.zip'
    urllib.request.urlretrieve(url, '/tmp/arc.zip')
    with zipfile.ZipFile('/tmp/arc.zip') as z:
        for name in z.namelist():
            if 'data/training/' in name and name.endswith('.json'):
                task_name = name.split('/')[-1]
                with z.open(name) as src, open(f'{ARC_DIR}/{task_name}', 'wb') as dst:
                    dst.write(src.read())
    print(f'Downloaded {len(os.listdir(ARC_DIR))} tasks.')
else:
    print(f'{len(os.listdir(ARC_DIR))} tasks already present.')

In [ ]:
# ── Cell 4: Generate descriptions ───────────────────────────────────────
# Calls Claude for each task not yet in the progress file.
# Safe to interrupt and re-run — resumes from where it left off.
#
# Change LIMIT to a small number (e.g. 5) to do a quick test run first.
import subprocess, sys

LIMIT = None   # set to an integer to process only that many tasks

cmd = [
    sys.executable,
    '/content/arc-agi-solver/scripts/generate_claude_descriptions.py',
    '--out-file', DESCRIPTIONS_FILE,
]
if LIMIT:
    cmd += ['--limit', str(LIMIT)]

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f'Exit code: {proc.returncode}')

In [ ]:
# ── Cell 5: Review sample descriptions ──────────────────────────────────
# Read the progress file and show N random examples so you can check quality.
import json, random

SAMPLE_SIZE = 3   # change to see more examples

with open(DESCRIPTIONS_FILE) as f:
    all_descs = json.load(f)

print(f'{len(all_descs)} descriptions in file.\n')

sample_ids = random.sample(list(all_descs), min(SAMPLE_SIZE, len(all_descs)))
for tid in sample_ids:
    print(f'{'='*60}')
    print(f'Task: {tid}')
    print(f'{'='*60}')
    print(all_descs[tid])
    print()

In [ ]:
# ── Cell 6: Copy descriptions into the repo and commit ──────────────────
# Copies the completed file from Drive into the cloned repo, then commits
# and pushes so it's available for the fine-tuning notebook.
#
# Only run this cell when all 400 tasks are done (or you want to save
# a partial snapshot).
import json, shutil, subprocess

REPO = '/content/arc-agi-solver'
DEST = f'{REPO}/data/claude_descriptions.json'

shutil.copy(DESCRIPTIONS_FILE, DEST)
n = len(json.loads(open(DEST).read()))
print(f'Copied {n} descriptions → {DEST}')

# data/ is gitignored so we need -f (force) to add files inside it.
# This is safe — the file is already tracked in the remote repo.
subprocess.run(['git', '-C', REPO, 'add', '-f', 'data/claude_descriptions.json'], check=True)
subprocess.run(['git', '-C', REPO, 'commit',
                '-m', f'Add Claude descriptions ({n} tasks)'], check=True)
subprocess.run(['git', '-C', REPO, 'push'], check=True)
print('Pushed to GitHub.')